In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- timeseries_extract ---
def _timeseries_param_pd():
    return SimpleNamespace(values=pd.DataFrame({"time": [0, 1, 2], "value": [1.0, 2.0, 3.0]}))
def _timeseries_param_pl():
    return SimpleNamespace(values=pl.DataFrame({"time": [0, 1, 2], "value": [1.0, 2.0, 3.0]}))
FIX_TIMESERIES_EXTRACT_RESULT_SERIES_PARAM_BEFORE = pd.Series([10, 20, 30], name="value")
FIX_TIMESERIES_EXTRACT_RESULT_SERIES_PARAM_GEN = pl.Series("value", [10, 20, 30])
FIX_TIMESERIES_EXTRACT_PARAM_DATA_BEFORE = _timeseries_param_pd()
FIX_TIMESERIES_EXTRACT_PARAM_DATA_GEN = _timeseries_param_pl()

# --- timeseries_gain ---
FIX_TIMESERIES_GAIN_ROW = {"id": 1, "value": "test"}
FIX_TIMESERIES_GAIN_OLD_PD = pd.DataFrame({"date": [1, 2], "A": [1.0, None], "B": [2.0, None], "C": [3.0, 5.0]})
FIX_TIMESERIES_GAIN_OLD_PL = pl.from_pandas(FIX_TIMESERIES_GAIN_OLD_PD)
FIX_TIMESERIES_GAIN_NEW_PD = pd.Series([4.0, 6.0], name="new")
FIX_TIMESERIES_GAIN_NEW_PL = pl.Series("new", [4.0, 6.0])

print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_timeseries_extract(param_data, result_series_param):
    param_data.values.loc[:, f"S{result_series_param.name}"] = result_series_param.values
    return None

def before_timeseries_gain(row):
    def gain_of_value_pairs(old_values: pd.DataFrame, new_values: pd.Series) -> float:
        old_score = old_values.apply(lambda row: row.dropna().size >= 4).sum()  # 5: dates plus 4 values
        old_values.loc[:, f"S{new_values.name}"] = new_values.values  # Add new column
        new_score = old_values.apply(lambda row: row.dropna().size >= 4).sum()  # 5: dates plus 4 values
    return gain_of_value_pairs

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_timeseries_extract(param_data, result_series_param):

    param_data.values = param_data.values.with_columns(
        pl.Series(f"S{result_series_param.name}", result_series_param.to_list())
    )
    return None

def gen_timeseries_gain(row):

    def gain_of_value_pairs(old_values: pl.DataFrame, new_values: pl.Series) -> float:
        if old_values.width < 4:
            old_score = 0
        else:
            old_score = old_values.select(
                (pl.sum_horizontal(pl.all().is_not_null().cast(pl.Int64)) >= 4)
                .cast(pl.Int64)
                .sum()
            ).item()

        old_values = old_values.with_columns(
            pl.Series(f"S{new_values.name}", new_values.to_numpy())
        )

        if old_values.width < 4:
            new_score = 0
        else:
            new_score = old_values.select(
                (pl.sum_horizontal(pl.all().is_not_null().cast(pl.Int64)) >= 4)
                .cast(pl.Int64)
                .sum()
            ).item()
    return gain_of_value_pairs

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: timeseries_extract ===

try:
    _param = _timeseries_param_pl()
    _r = gen_timeseries_extract(_param, FIX_TIMESERIES_EXTRACT_RESULT_SERIES_PARAM_GEN)
    print("✅ L1 smoke gen_timeseries_extract: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_timeseries_extract: {type(_e).__name__}: {_e}")

try:
    _param = _timeseries_param_pd()
    _rb = before_timeseries_extract(_param, FIX_TIMESERIES_EXTRACT_RESULT_SERIES_PARAM_BEFORE)
    print("✅ L1 smoke before_timeseries_extract: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_timeseries_extract: {type(_e).__name__}: {_e}")

try:
    _before_param = _timeseries_param_pd()
    _gen_param = _timeseries_param_pl()
    before_timeseries_extract(_before_param, FIX_TIMESERIES_EXTRACT_RESULT_SERIES_PARAM_BEFORE)
    gen_timeseries_extract(_gen_param, FIX_TIMESERIES_EXTRACT_RESULT_SERIES_PARAM_GEN)
    compare(_before_param.values, _gen_param.values, "timeseries_extract mutated values")
except Exception as _e:
    print(f"❌ L2 equivalence timeseries_extract: setup error — {type(_e).__name__}: {_e}")

try:
    _empty_before = SimpleNamespace(values=pd.DataFrame({"time": pd.Series(dtype="int64"), "value": pd.Series(dtype="float64")}))
    _empty_gen = SimpleNamespace(values=pl.DataFrame(schema={"time": pl.Int64, "value": pl.Float64}))
    before_timeseries_extract(_empty_before, pd.Series([], name="value", dtype="int64"))
    gen_timeseries_extract(_empty_gen, pl.Series("value", [], dtype=pl.Int64))
    compare(_empty_before.values, _empty_gen.values, "L3 edge timeseries_extract empty", check_row_order=True)
except Exception as _e:
    print(f"❌ L3 edge timeseries_extract empty: {type(_e).__name__}: {_e}")

try:
    _before_param = SimpleNamespace(values=pd.DataFrame({"time": [0, 1], "Svalue": [9.0, 9.0]}))
    _gen_param = SimpleNamespace(values=pl.DataFrame({"time": [0, 1], "Svalue": [9.0, 9.0]}))
    _before_series = pd.Series([1.5, 2.5], name="value")
    _gen_series = pl.Series("value", [1.5, 2.5])
    before_timeseries_extract(_before_param, _before_series)
    gen_timeseries_extract(_gen_param, _gen_series)
    _left = pl.from_pandas(_before_param.values.reset_index(drop=True))
    pl_assert_frame_equal(_left, _gen_param.values, check_dtypes=False, check_row_order=False)
    print("✅ L3 edge timeseries_extract overwrite Svalue: MATCH")
except Exception as _e:
    print(f"❌ L3 edge timeseries_extract overwrite: {type(_e).__name__}: {_e}")
